## 4. COCO 2017 데이터셋 다운로드
- 빠른 검증/테스트: `!bash scripts/download_coco.sh` (Val 5,000장 + Annotations)
- 300 에포크 본격 학습: `!bash scripts/download_coco.sh --full` (Train 118,287장 포함, ~19GB)

## 1. 하드웨어 가속기(GPU/TPU) 확인

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("GPU가 감지되지 않았습니다. TPU 또는 CPU 모드로 실행됩니다.")

## 2. Google Drive 마운트 (체크포인트 영구 저장)
학습 중 세션이 끊기더라도 가중치가 소실되지 않도록 구글 드라이브에 저장 디렉터리를 연결합니다.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/CNNResearch/checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"체크포인트 저장 경로: {DRIVE_CKPT_DIR}")

## 3. 저장소 클론 및 패키지 설치

In [ ]:
!git clone https://github.com/sonjuhy/CNNResearch.git
%cd CNNResearch
!pip install -q -r requirements.txt

## 4. COCO 2017 데이터셋 다운로드
- 빠른 검증/테스트: `!bash scripts/download_coco.sh` (Val 5,000장 + Annotations)
- 300 에포크 본격 학습: `!bash scripts/download_coco.sh --full` (Train 118,287장 포함, ~19GB)

In [ ]:
# 빠른 실행용 (Val2017 + Annotations)
!bash scripts/download_coco.sh

# 본격 300 에포크 학습 시 아래 주석을 해제하세요:
# !bash scripts/download_coco.sh --full

## 5. TensorBoard 실시간 모니터링

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/sota_detector

## 6. 모델 학습 (Training)
구글 드라이브 경로로 체크포인트를 실시간 저장합니다.

In [ ]:
# [GPU 기본 학습 - Nano 스케일]
!python train.py \n    --config configs/nano.yaml \n    --checkpoint-dir "/content/drive/MyDrive/CNNResearch/checkpoints" \n    --workers 4

# [TPU 환경 학습 시 (--tpu 플래그 추가)]
# !python train.py --config configs/nano.yaml --tpu --checkpoint-dir "/content/drive/MyDrive/CNNResearch/checkpoints"

# [세션 끊김 후 이어 학습하기 (Resume)]
# !python train.py --config configs/nano.yaml --resume "/content/drive/MyDrive/CNNResearch/checkpoints/last_nano.pt" --checkpoint-dir "/content/drive/MyDrive/CNNResearch/checkpoints"

## 7. mAP 평가 (Evaluation)

In [ ]:
!python eval.py \n    --scale nano \n    --weights "/content/drive/MyDrive/CNNResearch/checkpoints/best_nano.pt"

## 8. ONNX 및 Int8 PTQ 양자화 모델 배포

In [ ]:
# ONNX 모델 변환
!python export.py \n    --scale nano \n    --weights "/content/drive/MyDrive/CNNResearch/checkpoints/best_nano.pt"

# NPU/Edge용 Int8 양자화
!python utils/quantize.py